<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/FGF_gasket.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK COMPLET (propre) — ET_clean_v1
# Cell-1 -> Résultat final (multi-niveaux) avec:
#   L0 = 320
#   p_stay = 0.0  (marche non-lazy)
# Objectif:
#   - Tessellation 4-connexe (adjacence géométrique, pas de diagonales)
#   - d_h, d_s, d_w + versions scannées d_h*, d_s*, d_w*
#   - Coarse-graining 2×2 multi-niveaux (Appendix C.6 style)
# ============================================================

# -------------------------
# Cell 1 — Imports
# -------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x


# -------------------------
# Cell 2 — Version / helpers
# -------------------------
ET_VERSION = "ET_clean_v1_L0_320_pstay_0"
print("Loaded:", ET_VERSION)

def ET_build_adj_from_edges(n_nodes, edges, undirected=True, remove_self_loops=True):
    neigh = [[] for _ in range(n_nodes)]
    for u, v in edges:
        if remove_self_loops and u == v:
            continue
        neigh[u].append(v)
        if undirected:
            neigh[v].append(u)
    adj = []
    for u in range(n_nodes):
        if not neigh[u]:
            adj.append(np.array([], dtype=np.int32))
        else:
            adj.append(np.unique(np.array(neigh[u], dtype=np.int32)))
    return adj

def ET_edge_count_undirected(adj):
    return int(sum(len(a) for a in adj) // 2)


# -------------------------
# Cell 3 — Tessellation: grille 4-connexe (G_ell)
# -------------------------
def ET_make_grid_4conn(L, periodic=True):
    """
    Sommets = cellules; arêtes = bords partagés (4-voisins); pas de diagonales.
    periodic=True -> tore.
    """
    n = L * L
    def idx(i, j):
        return (i % L) * L + (j % L)

    edges = []
    for i in range(L):
        for j in range(L):
            u = idx(i, j)
            # droite + bas (undirected)
            if periodic or (j + 1 < L):
                edges.append((u, idx(i, j + 1)))
            if periodic or (i + 1 < L):
                edges.append((u, idx(i + 1, j)))
    return ET_build_adj_from_edges(n, edges, undirected=True)


# -------------------------
# Cell 4 — d_h: volumes de boules (BFS)
# -------------------------
def ET_ball_volume_counts(adj, center, r_max):
    dist = {center: 0}
    q = deque([center])
    shell = np.zeros(r_max + 1, dtype=np.int64)
    shell[0] = 1
    while q:
        u = q.popleft()
        du = dist[u]
        if du == r_max:
            continue
        for vv in adj[u]:
            v = int(vv)
            if v not in dist:
                dv = du + 1
                dist[v] = dv
                if dv <= r_max:
                    shell[dv] += 1
                    q.append(v)
    V = np.cumsum(shell)
    r = np.arange(1, r_max + 1)
    return r, V[1:].astype(np.float64)

def ET_volume_series(adj, r_max=60, n_centers=48, seed=0):
    rng = np.random.default_rng(seed)
    deg = np.array([len(a) for a in adj], dtype=int)
    valid = np.where(deg > 0)[0]
    centers = rng.choice(valid, size=min(n_centers, valid.size), replace=False)

    Vs = []
    for c in tqdm(centers, desc="Volume balls (BFS)"):
        r, V = ET_ball_volume_counts(adj, int(c), r_max)
        Vs.append(V)
    Vmean = np.mean(np.stack(Vs, axis=0), axis=0)
    return r, Vmean


# -------------------------
# Cell 5 — Random walk (non-lazy ici) + p_return + MSD
# -------------------------
def ET_random_walk(adj, start_nodes, T, p_stay=0.0, rng=None):
    """
    Marche aléatoire:
      - p_stay=0.0 => non-lazy
      - sinon lazy possible
    """
    if rng is None:
        rng = np.random.default_rng(0)

    start_nodes = np.asarray(start_nodes, dtype=np.int32)
    n_walkers = start_nodes.size
    traj = np.empty((T + 1, n_walkers), dtype=np.int32)
    traj[0] = start_nodes
    pos = start_nodes.copy()

    deg = np.array([len(a) for a in adj], dtype=np.int32)
    for t in range(1, T + 1):
        stay = rng.random(n_walkers) < p_stay if p_stay > 0 else np.zeros(n_walkers, dtype=bool)
        move_idx = np.where(~stay)[0]
        for k in move_idx:
            u = int(pos[k])
            d = int(deg[u])
            if d == 0:
                continue
            j = rng.integers(0, d)
            pos[k] = adj[u][j]
        traj[t] = pos
    return traj

def ET_return_series(adj, T=1200, n_centers=48, n_walkers_per_center=256, p_stay=0.0, seed=0):
    rng = np.random.default_rng(seed)
    deg = np.array([len(a) for a in adj], dtype=int)
    valid = np.where(deg > 0)[0]
    centers = rng.choice(valid, size=min(n_centers, valid.size), replace=False)

    p_acc = np.zeros(T + 1, dtype=np.float64)
    for c in tqdm(centers, desc="Return prob (RW)"):
        starts = np.full(n_walkers_per_center, int(c), dtype=np.int32)
        traj = ET_random_walk(adj, starts, T=T, p_stay=p_stay, rng=rng)
        p_acc += (traj == starts[None, :]).mean(axis=1)

    p = p_acc / max(len(centers), 1)
    t = np.arange(1, T + 1)
    return t, p[1:]

def ET_msd_series(adj, T=1200, n_centers=48, n_walkers_per_center=256, p_stay=0.0, seed=0):
    rng = np.random.default_rng(seed)
    n = len(adj)
    deg = np.array([len(a) for a in adj], dtype=int)
    valid = np.where(deg > 0)[0]
    centers = rng.choice(valid, size=min(n_centers, valid.size), replace=False)

    msd_acc = np.zeros(T + 1, dtype=np.float64)
    for c0 in tqdm(centers, desc="MSD (RW + BFS dist)"):
        c0 = int(c0)

        dist = np.full(n, -1, dtype=np.int32)
        dist[c0] = 0
        q = deque([c0])
        while q:
            u = q.popleft()
            du = dist[u]
            for vv in adj[u]:
                v = int(vv)
                if dist[v] < 0:
                    dist[v] = du + 1
                    q.append(v)

        starts = np.full(n_walkers_per_center, c0, dtype=np.int32)
        traj = ET_random_walk(adj, starts, T=T, p_stay=p_stay, rng=rng)
        msd_acc += (dist[traj] ** 2).mean(axis=1)

    msd = msd_acc / max(len(centers), 1)
    t = np.arange(1, T + 1)
    return t, msd[1:]


# -------------------------
# Cell 6 — Régression log-log + scan
# -------------------------
def ET_linreg_loglog(x, y):
    X = np.log(np.asarray(x, dtype=np.float64))
    Y = np.log(np.asarray(y, dtype=np.float64))
    A = np.vstack([np.ones_like(X), X]).T
    coef, *_ = np.linalg.lstsq(A, Y, rcond=None)
    a, b = coef
    Yhat = A @ coef
    ss_res = np.sum((Y - Yhat) ** 2)
    ss_tot = np.sum((Y - Y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return float(a), float(b), float(r2)

def ET_scan_windows(x, y, mins, maxs, min_pts=40, y_min=1e-18):
    x = np.asarray(x)
    y = np.asarray(y)
    rows = []
    for xmin in mins:
        for xmax in maxs:
            if xmax <= xmin:
                continue
            m = (x >= xmin) & (x <= xmax) & (y > y_min)
            if np.sum(m) < min_pts:
                continue
            a, b, r2 = ET_linreg_loglog(x[m], y[m])
            rows.append((int(xmin), int(xmax), float(b), float(r2), int(np.sum(m))))
    rows.sort(key=lambda z: (-(z[3] if np.isfinite(z[3]) else -np.inf), -(z[1] - z[0])))
    return rows


# -------------------------
# Cell 7 — Borne anti taille finie
# -------------------------
def ET_tmax_upper_from_N(N_nodes, T_used):
    """
    Heuristique anti plateau de mélange.
    L0=320 => niveaux plus grands, donc on peut garder cette borne.
    """
    return int(min(T_used, max(200, 0.03 * N_nodes)))


# -------------------------
# Cell 8 — Analyse complète d'un graphe: fixed(borné) + star(scan)
# -------------------------
def ET_analyze_graph(adj, r_max=80, T=1200,
                     n_centers=48, n_walkers_per_center=256,
                     p_stay=0.0, seed=123, do_plots=False):
    N = len(adj)
    tmax_upper = ET_tmax_upper_from_N(N, T)

    # séries
    r, V = ET_volume_series(adj, r_max=r_max, n_centers=n_centers, seed=seed)
    tR, pR = ET_return_series(adj, T=T, n_centers=n_centers, n_walkers_per_center=n_walkers_per_center,
                              p_stay=p_stay, seed=seed + 1)
    tM, msd = ET_msd_series(adj, T=T, n_centers=n_centers, n_walkers_per_center=n_walkers_per_center,
                            p_stay=p_stay, seed=seed + 2)

    # fixed windows (bornées)
    t_fit_min = 50
    t_fit_max = min(500, tmax_upper)

    r_fit_min = max(6, min(12, r_max - 10))
    r_fit_max = max(r_fit_min + 8, min(45, r_max))

    m = (r >= r_fit_min) & (r <= r_fit_max) & (V > 0)
    _, dh_fixed, _ = ET_linreg_loglog(r[m], V[m])

    m = (tR >= t_fit_min) & (tR <= t_fit_max) & (pR > 0)
    a_s, b_s, r2_s = ET_linreg_loglog(tR[m], pR[m])
    ds_fixed = -2.0 * b_s

    m = (tM >= t_fit_min) & (tM <= t_fit_max) & (msd > 0)
    a_w, b_w, r2_w = ET_linreg_loglog(tM[m], msd[m])
    dw_fixed = 2.0 / b_w if b_w > 0 else np.nan

    coh_fixed = 2.0 * dh_fixed / dw_fixed if (np.isfinite(dw_fixed) and dw_fixed > 0) else np.nan

    # scanned (*)
    tmins = range(30, max(60, tmax_upper - 80) + 1, 10)
    tmaxs = range(max(120, int(0.6 * tmax_upper)), tmax_upper + 1, 20)

    rows_s = ET_scan_windows(tR, pR, tmins, tmaxs, min_pts=50)
    if rows_s:
        _, _, b_s_star, r2_s_star, _ = rows_s[0]
        ds_star = -2.0 * b_s_star
    else:
        ds_star, r2_s_star = np.nan, np.nan

    rows_w = ET_scan_windows(tM, msd, tmins, tmaxs, min_pts=50)
    if rows_w:
        _, _, b_w_star, r2_w_star, _ = rows_w[0]
        dw_star = 2.0 / b_w_star if b_w_star > 0 else np.nan
    else:
        dw_star, r2_w_star = np.nan, np.nan

    best = None
    for rmin in range(4, max(6, r_max - 12) + 1):
        for rmax2 in range(max(12, r_max - 20), r_max + 1):
            if rmax2 <= rmin + 7:
                continue
            mm = (r >= rmin) & (r <= rmax2) & (V > 0)
            if np.sum(mm) < 12:
                continue
            a, b, r2 = ET_linreg_loglog(r[mm], V[mm])
            cand = (rmin, rmax2, b, r2)
            if (best is None) or (cand[3] > best[3]) or (cand[3] == best[3] and (cand[1]-cand[0] > best[1]-best[0])):
                best = cand
    if best is not None:
        _, _, dh_star, r2_h_star = best
    else:
        dh_star, r2_h_star = np.nan, np.nan

    coh_star = 2.0 * dh_star / dw_star if (np.isfinite(dw_star) and dw_star > 0) else np.nan

    if do_plots:
        plt.figure(); plt.loglog(r, V, marker="o", linestyle="none"); plt.xlabel("r"); plt.ylabel("V(r)"); plt.show()
        plt.figure(); plt.loglog(tR, pR, marker="o", linestyle="none"); plt.xlabel("t"); plt.ylabel("p_ret(t)"); plt.show()
        plt.figure(); plt.loglog(tM, msd, marker="o", linestyle="none"); plt.xlabel("t"); plt.ylabel("MSD(t)"); plt.show()

    return {
        "N_nodes": N,
        "tmax_upper": int(tmax_upper),
        "t_fit_max_used": int(t_fit_max),
        "fixed": {
            "d_h": float(dh_fixed),
            "d_s": float(ds_fixed),
            "d_w": float(dw_fixed),
            "coh": float(coh_fixed),
            "r2_ds": float(r2_s),
            "r2_dw": float(r2_w),
        },
        "star": {
            "d_h": float(dh_star),
            "d_s": float(ds_star),
            "d_w": float(dw_star),
            "coh": float(coh_star),
            "r2_dh": float(r2_h_star),
            "r2_ds": float(r2_s_star),
            "r2_dw": float(r2_w_star),
        }
    }


# -------------------------
# Cell 9 — Coarse-graining 2×2 multi-niveaux
# -------------------------
def ET_partition_grid_block(L, block=2):
    assert L % block == 0
    part = np.empty(L * L, dtype=np.int32)
    Lc = L // block
    for i in range(L):
        for j in range(L):
            part[i * L + j] = (i // block) * Lc + (j // block)
    return part, Lc

def ET_coarse_grain_by_partition(adj, part):
    N = len(adj)
    K = int(part.max()) + 1
    acc = [set() for _ in range(K)]
    for u in range(N):
        bu = int(part[u])
        for vv in adj[u]:
            bv = int(part[int(vv)])
            if bv != bu:
                acc[bu].add(bv)
    return [np.array(sorted(s), dtype=np.int32) for s in acc]

def ET_run_levels_grid_coarse_grain(L0=320, periodic=True, n_levels=2, block=2,
                                    r_max=80, T=1200, n_centers=48, n_walkers_per_center=256,
                                    p_stay=0.0, seed=123):
    L = L0
    adj = ET_make_grid_4conn(L=L, periodic=periodic)
    rows = []

    for level in range(n_levels + 1):
        r_max_lvl = min(r_max, max(18, int(0.35 * L)))  # un peu plus large pour L0=320
        out = ET_analyze_graph(
            adj, r_max=r_max_lvl, T=T,
            n_centers=n_centers, n_walkers_per_center=n_walkers_per_center,
            p_stay=p_stay, seed=seed + 1000 * level, do_plots=False
        )

        rows.append({
            "level": level,
            "L_grid": L,
            "N_nodes": len(adj),
            "E_edges": ET_edge_count_undirected(adj),
            "tmax_upper": out["tmax_upper"],
            "t_fit_max_used": out["t_fit_max_used"],
            # fixed (borné)
            "d_h_fixed": out["fixed"]["d_h"],
            "d_s_fixed": out["fixed"]["d_s"],
            "d_w_fixed": out["fixed"]["d_w"],
            "2dh/dw_fixed": out["fixed"]["coh"],
            "ds_fixed_R2": out["fixed"]["r2_ds"],
            "dw_fixed_R2": out["fixed"]["r2_dw"],
            # star (scan)
            "d_h*": out["star"]["d_h"],
            "d_s*": out["star"]["d_s"],
            "d_w*": out["star"]["d_w"],
            "2dh*/dw*": out["star"]["coh"],
            "dh*_R2": out["star"]["r2_dh"],
            "ds*_R2": out["star"]["r2_ds"],
            "dw*_R2": out["star"]["r2_dw"],
        })

        if level < n_levels:
            part, Lc = ET_partition_grid_block(L, block=block)
            adj = ET_coarse_grain_by_partition(adj, part)
            L = Lc

    df = pd.DataFrame(rows)
    display(df)
    return df


# ============================================================
# Cell 10 — RUN FINAL (L0=320, p_stay=0.0)
# niveaux: 0 -> 1 -> 2 (320 -> 160 -> 80)
# ============================================================
df = ET_run_levels_grid_coarse_grain(
    L0=320,
    periodic=True,
    n_levels=2,
    block=2,
    r_max=80,
    T=1200,
    n_centers=48,
    n_walkers_per_center=256,
    p_stay=0.0,
    seed=123
)
df